# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa" using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset provides survey data focused on mental health in Kilifi County, Kenya. We will:

1. Load Croissant metadata and records
2. Explore available record sets, fields, and their `@id`s
3. Load specific data into DataFrames
4. Conduct exploratory data analysis (EDA)
5. Visualize some results

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading

We will load the metadata and records from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"Version: {meta.version}")
print(f"Keywords: {meta.keywords}")

## 2. Data Overview

List record sets, their fields, and their `@id`s for reference. We use the Croissant schema structure for exploration. All entities (record sets, fields, and columns) are referenced by their `@id` as per best practice.

In [ ]:
# Get all record set @id's
record_sets = list(dataset.metadata.recordSets.keys())
print("Available record sets in the dataset (@id):")
for rs_id in record_sets:
    record_set_obj = dataset.metadata.recordSets[rs_id]
    print(f"- {rs_id}: {record_set_obj.name if hasattr(record_set_obj, 'name') else ''}")
    # List fields within this record set
    if hasattr(record_set_obj, 'fields'):
        print("  Fields:")
        for field_id, field_obj in record_set_obj.fields.items():
            print(f"    - {field_id}: {field_obj.name if hasattr(field_obj, 'name') else ''}")
    print()

## 3. Data Extraction

Extract data records from a chosen record set and load to DataFrames for analysis. Use the `@id`s above to select the main survey and ancillary tables if present.

Below, we assemble all record sets into DataFrames indexed by their `@id` for flexible exploration.

In [ ]:
# List chosen record set IDs for extraction. Replace the following with the IDs you observed in section 2.
main_record_sets = record_sets  # extract all available for demonstration

dataframes = {}

for rs_id in main_record_sets:
    try:
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for {rs_id} with shape {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set {rs_id}.")
    except Exception as e:
        print(f"Error loading records from {rs_id}: {e}")

# For further analysis, we'll pick the main record set (assume the first in the list is main)
if main_record_sets:
    main_rs_id = main_record_sets[0]
    print(f"\nPreview of records in main record set: {main_rs_id}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

We process numeric fields (for example, GAD-7, PHQ-9, PSQ total scores) by filtering, normalizing, and grouping records. Replace `gad7_total`, `phq9_total`, etc., by their exact `@id` as listed above. This code demonstrates selection, filtering, normalization, and grouping.

If you know the exact field `@id` for a numeric field (e.g., GAD-7 score), place it below (see the printed columns above).

In [ ]:
# Example, adjust these field names according to overview section output
numeric_field_id = None
group_field_id = None
main_df = dataframes[main_rs_id]

# Identify numeric fields automatically
num_field_found = False
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        num_field_found = True
        break
if not num_field_found:
    print("No numeric columns found for analysis. Please check field mapping above.")
else:
    print(f"Using numeric field: {numeric_field_id}")

    # Filtering records where the value is greater than a threshold
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].mean() > 0 else 1
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} (first 5 rows):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a likely demographic field (e.g., level_of_education or gender), select the first object type field
    for col in main_df.columns:
        if pd.api.types.is_object_dtype(main_df[col]) and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id} (first 5 groups):")
        display(grouped_df.head())

## 5. Visualization

Let's plot the distribution of a numeric mental health score and how it differs by the chosen demographic field (if available).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if numeric field was found
if numeric_field_id is not None:
    plt.figure(figsize=(10, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We have loaded the FAIR² Kilifi County Mental Health Survey dataset using the Croissant standard and `mlcroissant`.
- All records, fields, and columns were referenced and extracted using their `@id`.
- Example EDA steps included numeric filtering, normalization, grouping, and basic visualization.

To continue, tailor field selection, outlier handling, and visualizations according to your analysis needs and based on your inspection of available field `@id`s listed in the overview. For more, see the [mlcroissant documentation](https://croissant.mlcommons.org/).
